# 03 — Temporal Validation Strategy

Ce notebook est consacré à la construction d'une stratégie de validation temporelle pour le challenge QRT Asset Allocation Performance Forecasting.

L'objectif n'est pas encore d'entraîner des modèles, mais de définir un protocole fiable permettant d'évaluer les futurs modèles dans des conditions proches du test officiel.

## Objectif

Les données du challenge étant organisées chronologiquement, une validation classique par mélange aléatoire introduirait un biais temporel.

L'objectif de ce notebook est de construire une stratégie de validation respectant l'ordre des observations grâce à une validation chronologique en fenêtre d'entraînement croissante (*expanding window*). Cette stratégie sera utilisée de manière identique pour l'ensemble des modèles développés dans le projet.

## Rôle de `TS`

La variable `TS` représente l'ordre temporel des observations.

Elle est centrale pour construire les splits de validation, car le découpage ne doit pas être fait ligne par ligne, mais date par date.

Toutes les observations appartenant à une même date doivent rester dans le même bloc : soit dans le train local, soit dans la validation locale.

## 0. Configuration du chemin projet

In [5]:
import sys 
from pathlib import Path

ROOT = Path.cwd().parent

if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.data_loading import load_X_train, load_X_test, load_y_train, load_sample_submission

## Chargement des données

Nous chargeons les fichiers d'entraînement, de test et la cible.

À ce stade, aucune transformation de features n'est effectuée. Le but est uniquement d'auditer la structure temporelle des données.

In [7]:
X_train = load_X_train()
X_test = load_X_test()
y_train = load_y_train()
y_test = load_sample_submission()

## 1. Audit temporel des données

Avant de construire un holdout ou des folds temporels, nous devons vérifier que la colonne `TS` fournit un ordre temporel exploitable.

Nous allons vérifier :

- le nombre de dates uniques ;
- la première date disponible ;
- la dernière date disponible ;
- l'ordre des dates ;
- la continuité des dates ;
- le nombre d'observations par date.

### Nombre de dates uniques

Nous commençons par compter le nombre de dates distinctes dans le train.

Cette information permet de connaître la profondeur historique disponible pour construire une validation temporelle.

In [16]:
X_train["TS"].nunique()

2522

### Bornes temporelles du train

Nous identifions ensuite la première et la dernière date du train.

Ces bornes permettent de vérifier que le train couvre bien une période ancienne qui précède le test officiel.

In [6]:
X_train.sort_values("TS",ascending=True)["TS"].head(1)

0    DATE_0001
Name: TS, dtype: str

In [7]:
X_train.sort_values("TS",ascending=False)["TS"].head(1)

527072    DATE_2522
Name: TS, dtype: str

Le train couvre la période allant de `DATE_0001` à `DATE_2522`.

Cette structure confirme que le train contient un historique ordonné, qui pourra être découpé en train local et validation locale.

### Tri des dates

Nous trions les observations par `TS` afin de respecter l'ordre temporel.

Le tri est indispensable avant toute construction de split temporel.

In [9]:
X_train_sorted_TS = X_train.sort_values("TS",ascending=True)

Le dataset est maintenant ordonné chronologiquement selon `TS`.

Cependant, trier les lignes ne suffit pas : la validation devra être construite à partir des dates uniques, afin d'éviter qu'une même date soit présente à la fois dans le train local et la validation locale.

### Créons une liste qui contient les dates uniques de notre jeu d'entrainement

In [18]:
unique_dates_sorted = list(X_train_sorted_TS["TS"].unique())
print(f"La première date de notre ensemble de données d'entraînement est : {unique_dates_sorted[0]}")
print(f"La dernière date de notre ensemble de données d'entraînement est : {unique_dates_sorted[-1]}")
print(f"Le nombre de dates uniques de notre ensemble d'entraînement est : {len(unique_dates_sorted)}")

La première date de notre ensemble de données d'entraînement est : DATE_0001
La dernière date de notre ensemble de données d'entraînement est : DATE_2522
Le nombre de dates uniques de notre ensemble d'entraînement est : 2522


Sous l'hypothèse que les identifiants suivent une numérotation continue,
le fait d'observer 2522 dates uniques entre DATE_0001 et DATE_2522 suggère
qu'aucune date n'est manquante.

### Nombre d'observations par date

Nous analysons maintenant le nombre de lignes disponibles pour chaque date.

Cette vérification permet de comprendre si chaque période temporelle contient le même nombre d'allocations ou si la taille des dates varie dans le temps.

Cette information est importante pour l'interprétation des futurs scores de validation : une période contenant peu d'observations peut produire un score plus instable.

In [25]:
dates = X_train_sorted_TS.groupby("TS").count()
dates.describe()["ROW_ID"]

count    2522.000000
mean      208.990087
std        81.903327
min        65.000000
25%       140.000000
50%       276.000000
75%       276.000000
max       276.000000
Name: ROW_ID, dtype: float64

La taille des dates varie dans le temps. Cette variation est importante pour l’interprétation des scores de validation.

Une période de validation contenant moins d’observations produit une estimation de performance plus bruitée qu’une période contenant beaucoup d’observations.

## 2. Premier holdout temporel

Nous construisons maintenant un premier découpage temporel local.

L'idée est de simuler le test officiel à l'intérieur du train : le modèle apprend sur les dates anciennes et est évalué sur les dates les plus récentes du train.

Comme le test officiel contient environ 120 dates, nous utilisons les 120 dernières dates du train comme validation locale.

In [44]:
valid_dates = unique_dates_sorted[-120:]
train_dates = unique_dates_sorted[:-120]

In [51]:
full_train_df = X_train_sorted_TS.merge(y_train,left_on='ROW_ID',right_on='ROW_ID')
full_train_df

,ROW_ID,TS,ALLOCATION,RET_20,RET_19,RET_18,RET_17,RET_16,RET_15,RET_14,RET_13,RET_12,RET_11,RET_10,RET_9,RET_8,RET_7,RET_6,RET_5,RET_4,RET_3,RET_2,RET_1,SIGNED_VOLUME_20,SIGNED_VOLUME_19,SIGNED_VOLUME_18,SIGNED_VOLUME_17,SIGNED_VOLUME_16,SIGNED_VOLUME_15,SIGNED_VOLUME_14,SIGNED_VOLUME_13,SIGNED_VOLUME_12,SIGNED_VOLUME_11,SIGNED_VOLUME_10,SIGNED_VOLUME_9,SIGNED_VOLUME_8,SIGNED_VOLUME_7,SIGNED_VOLUME_6,SIGNED_VOLUME_5,SIGNED_VOLUME_4,SIGNED_VOLUME_3,SIGNED_VOLUME_2,SIGNED_VOLUME_1,MEDIAN_DAILY_TURNOVER,GROUP,target
0,0,DATE_0001,ALLOCATION_01,-0.018192,-0.000306,-0.006881,-0.002393,0.000507,-0.001270,-0.002539,0.002830,0.005398,-0.000009,-0.009898,0.001350,0.003312,-0.009116,0.002864,-0.009067,0.001514,0.001013,-0.000178,0.003944,1.208587,1.000000,0.983866,0.712591,0.691989,0.865593,0.990404,0.876563,0.700953,0.594470,0.381315,-0.172996,0.818730,0.941014,0.714129,-0.323847,0.525097,0.363601,-0.219328,NaN,0.096905,1,0.009210
1,175,DATE_0001,ALLOCATION_251,-0.006551,-0.000907,-0.004108,0.000940,0.000437,-0.002351,-0.001574,0.001946,-0.001762,-0.001745,-0.004506,0.001152,0.000168,-0.003338,0.002516,-0.005296,-0.001486,0.000752,-0.001194,-0.000461,1.150083,0.939604,1.661606,1.053446,1.591433,1.628179,1.987375,1.218436,0.822772,0.735983,0.782848,0.711655,1.395965,1.442158,1.351921,0.541005,0.758406,0.972364,0.603374,NaN,0.052581,1,0.003776
2,176,DATE_0001,ALLOCATION_252,-0.016768,-0.007595,-0.002731,0.009537,0.007069,-0.005130,-0.004918,-0.001318,0.002822,-0.000183,-0.004474,0.007539,-0.000142,-0.008985,0.004702,-0.005207,-0.000598,-0.002637,-0.002020,-0.000944,1.185678,0.766598,1.128923,1.053374,1.293490,0.973119,1.391784,1.281552,1.063922,1.091127,0.949998,1.352597,1.321167,0.865953,1.561624,0.964161,0.993197,0.635015,0.672014,NaN,0.010366,1,0.007062
3,177,DATE_0001,ALLOCATION_253,0.000629,-0.002134,0.002005,-0.001176,-0.000579,-0.000900,0.000710,0.001244,-0.000945,0.000488,0.000714,-0.000103,0.000837,-0.000861,0.002631,-0.000442,0.001745,0.000693,-0.001329,-0.000125,-1.000000,-1.333956,-1.000000,-1.248037,-1.137327,-0.892823,-0.742567,-0.867726,-1.539042,-1.636965,-1.999707,-1.382821,-1.189733,-0.923068,-1.128905,-0.889498,-1.226147,-1.163023,-0.987666,NaN,0.016445,4,-0.000149
4,178,DATE_0001,ALLOCATION_254,-0.005754,-0.003239,-0.002696,0.000174,0.002123,-0.002094,-0.001061,0.002554,-0.000123,0.002797,-0.003843,0.001724,0.001306,-0.001118,0.000621,-0.002930,-0.003306,-0.000962,-0.002302,0.000217,-5.418186,-3.502813,-1.236900,-1.412135,-0.147055,-1.584888,-1.250179,-1.424552,-1.068404,-0.830219,0.335588,-0.455320,0.875391,-1.879517,-1.275797,-0.385431,0.659207,1.261280,0.292028,NaN,0.001886,1,0.000733
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
527068,526885,DATE_2522,ALLOCATION_172,-0.002032,-0.000054,0.000836,0.002466,0.000495,0.002860,0.000861,-0.001369,0.002064,-0.001419,0.002082,-0.000191,-0.000818,0.000621,-0.002931,0.000559,0.000124,0.000788,-0.000759,0.000384,0.892034,0.577764,1.010971,0.946092,1.014363,1.069182,1.666162,1.553756,1.052931,1.012377,1.188789,0.409310,0.693568,1.263960,1.173826,0.795431,1.175767,1.004167,1.175447,1.322866,0.082922,2,-0.000724
527069,526884,DATE_2522,ALLOCATION_171,0.003352,-0.000982,0.002896,0.003366,0.004442,-0.001910,0.001098,-0.002866,-0.005422,0.000085,-0.001571,0.001532,-0.002307,0.003917,0.002466,-0.002878,-0.003148,0.003570,0.004737,0.000634,-0.773689,-1.178214,-1.031182,-1.000000,-0.878054,-0.627872,-0.878095,-0.632570,-0.804830,-0.685162,-1.022529,-0.417299,-0.853222,-0.596505,-0.870762,-1.002471,-1.065143,-1.021265,-1.321066,NaN,0.032054,3,0.001296
527070,526883,DATE_2522,ALLOCATION_170,-0.000459,0.000918,0.001668,0.001066,-0.000849,0.000874,0.001105,0.000353,-0.001627,-0.000790,0.001577,0.001116,0.000451,0.000466,0.000526,0.001162,0.000391,-0.000881,-0.001992,0.000540,1.245961,-0.297997,2.501091,0.459970,0.505931,1.007243,1.364008,1.637892,1

In [52]:
df_train = full_train_df[full_train_df["TS"].isin(train_dates)]
df_validation = full_train_df[full_train_df["TS"].isin(valid_dates)]

In [63]:
unique_dates_train = set(df_train["TS"].unique())
unique_dates_validation = set(df_validation["TS"].unique())
print(f"Nombre de dates communes entre train et validation : {len(unique_dates_train & unique_dates_validation)}")
print(f"La derniere date du train est {max(unique_dates_train)} et la premiere date de l'evaluation est {min(unique_dates_validation)}")
print(f"nombre de dates train = {len(unique_dates_train)}")
print(f"nombre de dates validation = {len(unique_dates_validation)}")
print(f"Nombre de lignes du train = {len(df_train)}")
print(f"Nombre de lignes de la validation = {len(df_validation)}")

Nombre de dates communes entre train et validation : 0
La derniere date du train est DATE_2402 et la premiere date de l'evaluation est DATE_2403
nombre de dates train = 2402
nombre de dates validation = 120
Nombre de lignes du train = 501413
Nombre de lignes de la validation = 25660


Le holdout temporel respecte les contraintes attendues :

- aucune date n'est partagée entre le train local et la validation locale ;
- la dernière date du train local est `DATE_2402` ;
- la première date de validation locale est `DATE_2403` ;
- le train local contient 2402 dates ;
- la validation locale contient 120 dates.

Ce découpage reproduit la logique du challenge officiel : apprendre sur des dates anciennes puis évaluer sur des dates plus récentes.

In [56]:
X_train_holdout = df_train.drop("target",axis=1)
y_train_holdout = df_train[["ROW_ID","target"]]

In [57]:
X_validation_holdout = df_validation.drop("target",axis=1)
y_validation_holdout = df_validation[["ROW_ID","target"]]

### Limites d'un seul holdout

Ce holdout est méthodologiquement crédible, car il respecte l'ordre temporel et utilise une validation proche de la taille du test officiel.

Cependant, il ne donne qu'un seul score sur une seule période récente. Si cette période est atypique, le score peut surestimer ou sous-estimer la performance réelle du modèle.

C'est pourquoi nous construirons ensuite plusieurs folds temporels afin d'évaluer la stabilité des modèles sur plusieurs futurs historiques.

In [66]:
df_train["class"] = np.where(df_train["target"] > 0,1,0)
df_validation["class"] = np.where(df_validation["target"] > 0,1,0)
print(f"La proportion de classe positive dans le train est : {df_train["class"].mean() * 100:.2f}%")
print(f"La proportion de classe positive dans la validation est : {df_validation["class"].mean() * 100:.2f}%")

La proportion de classe positive dans le train est : 50.64%
La proportion de classe positive dans la validation est : 52.21%


## 3. Folds temporels à fenêtre croissante

Après avoir construit un premier holdout temporel, nous généralisons la logique à plusieurs folds.

L'objectif est de tester les futurs modèles sur plusieurs périodes de validation successives, et non sur une seule période récente.

Chaque fold respecte la chronologie : le train contient uniquement des dates antérieures à la validation.

### Principe de la fenêtre croissante

Dans une fenêtre croissante, le train commence toujours à la première date disponible et s'allonge progressivement à chaque fold.

La validation avance dans le temps par blocs de taille fixe.

Cette stratégie permet d'évaluer la stabilité temporelle des modèles tout en conservant un historique d'entraînement de plus en plus riche.

In [ ]:
def create_expanding_window_folds(dates:list[str],size:int=120,k:int=4) -> list:
    if (size <= 0) or (k <= 0) :
        raise ValueError("La taille du set de validation et le nombre de folds doivent etre positifs")
    if (len(dates) == 0) or (len(dates) <= k * size) :
        raise ValueError("Le nombre de dates est insuffisant !")
    if len(set(dates)) != len(dates) :
        raise ValueError("La liste des dates ne doit pas contenir de doublons !")
    dates = dates.sort()

    dict_enum = {}
    for i in range(0,len(dates)):
        dict_enum[dates[i]] = i

    train_start = dates[0]
    valid_end = dates[-1]
    folds = []
    for i in range(k,0,-1):
        fold_k = {}
        fold_k["fold"] = i
        fold_k["train_start"] = train_start
        indice_valid_start = dict_enum[valid_end] + 1 - size
        valid_start = dates[indice_valid_start]
        train_end = dates[indice_valid_start - 1]
        fold_k["train_end"] = train_end
        fold_k["valid_start"] = valid_start
        fold_k["valid_end"] = valid_end
        valid_end = dates[dict_enum[valid_end] - size]
        folds.append(fold_k)

    return folds[::-1]

In [ ]:
def check_temporal_folds(folds,dates,validation_size):
    for i in range(len(folds)):
        train_fold_dates = list(filter(lambda x: (x >= folds[i]["train_start"]) & (x <= folds[i]["train_end"]),dates))
        valid_fold_dates = list(filter(lambda x: (x >= folds[i]["valid_start"]) & (x <= folds[i]["valid_end"]),dates))
        if len(train_fold_dates) == 0:
            raise ValueError(f"L'ensemble d'entrainement du fold {i+1} est vide!")
        if len(valid_fold_dates) == 0:
            raise ValueError(f"L'ensemble de validation du fold {i+1} est vide!")
        if len(set(train_fold_dates) & set(valid_fold_dates)) != 0:
            raise ValueError("Le train et la validation ont des dates en commun !")
        if max(train_fold_dates) >= min(valid_fold_dates):
            raise ValueError(f"{max(train_fold_dates)} n'est pas inférieure à {min(valid_fold_dates)} !")
        if len(valid_fold_dates) != validation_size:
            raise ValueError(f"le nombre de dates du set de validation du fold {i+1} est différent de {validation_size}")
    return True

In [93]:
dates = full_train_df["TS"].unique()

In [96]:
create_expanding_window_folds(dates)

[{'fold': 1,
  'train_start': 'DATE_0001',
  'train_end': 'DATE_2042',
  'valid_start': 'DATE_2043',
  'valid_end': 'DATE_2162'},
 {'fold': 2,
  'train_start': 'DATE_0001',
  'train_end': 'DATE_2162',
  'valid_start': 'DATE_2163',
  'valid_end': 'DATE_2282'},
 {'fold': 3,
  'train_start': 'DATE_0001',
  'train_end': 'DATE_2282',
  'valid_start': 'DATE_2283',
  'valid_end': 'DATE_2402'},
 {'fold': 4,
  'train_start': 'DATE_0001',
  'train_end': 'DATE_2402',
  'valid_start': 'DATE_2403',
  'valid_end': 'DATE_2522'}]

### Contrôles de cohérence des folds

Après avoir généré les bornes temporelles des folds, nous vérifions que chaque découpage respecte les contraintes méthodologiques attendues.

Pour chaque fold, nous contrôlons :

- l'absence de dates communes entre train et validation ;
- le respect de l'ordre temporel ;
- la taille de la validation ;
- l'existence d'un historique d'entraînement avant la validation.

Ces contrôles permettent de limiter les risques de leakage temporel avant d'utiliser les folds pour entraîner des modèles.

In [ ]:
folds = create_expanding_window_folds(dates)

In [98]:
full_train_df["class"] = np.where(full_train_df["target"] > 0,1,0)

In [103]:
for i in range(len(folds)):
    print(f"*------------------ Folds {i+1} -------------------------*")
    train_dates = set(full_train_df[(full_train_df["TS"] >= folds[i]["train_start"]) & (full_train_df["TS"] <= folds[i]["train_end"])]["TS"].unique())
    validation_dates = set(full_train_df[(full_train_df["TS"] >= folds[i]["valid_start"]) & (full_train_df["TS"] <= folds[i]["valid_end"])]["TS"].unique())
    print(f"Nombre de dates communes entre le train et la validation : {len(train_dates & validation_dates)}")
    print(f"Max(train_dates) = {max(train_dates)} et Min(valid_dates) = {min(validation_dates)}")
    print(f"Le nombre de dates dans la validation du fold {i+1} est {len(validation_dates)}")
    print("\n\n")

*------------------ Folds 1 -------------------------*
Nombre de dates communes entre le train et la validation : 0
Max(train_dates) = DATE_2042 et Min(valid_dates) = DATE_2043
Le nombre de dates dans la validation du fold 1 est 120



*------------------ Folds 2 -------------------------*
Nombre de dates communes entre le train et la validation : 0
Max(train_dates) = DATE_2162 et Min(valid_dates) = DATE_2163
Le nombre de dates dans la validation du fold 2 est 120



*------------------ Folds 3 -------------------------*
Nombre de dates communes entre le train et la validation : 0
Max(train_dates) = DATE_2282 et Min(valid_dates) = DATE_2283
Le nombre de dates dans la validation du fold 3 est 120



*------------------ Folds 4 -------------------------*
Nombre de dates communes entre le train et la validation : 0
Max(train_dates) = DATE_2402 et Min(valid_dates) = DATE_2403
Le nombre de dates dans la validation du fold 4 est 120





### Interprétation des contrôles

Les quatre folds respectent les contraintes temporelles attendues.

Pour chaque fold, aucune date n'est partagée entre le train et la validation, et la dernière date du train précède strictement la première date de validation.

Chaque validation contient exactement 120 dates, ce qui permet de comparer les folds sur des périodes de même longueur.

Ces contrôles confirment que les folds peuvent être utilisés comme base de validation temporelle pour les futurs modèles.

In [110]:
for i in range(len(folds)) :
    folds[i]["nb_dates_train"] = len(dates[(dates >= folds[i]["train_start"]) & (dates <= folds[i]["train_end"])])
    folds[i]["nb_dates_validation"] = len(dates[(dates >= folds[i]["valid_start"]) & (dates <= folds[i]["valid_end"])])
    fold_train = full_train_df[(full_train_df["TS"] >= folds[i]["train_start"]) & (full_train_df["TS"] <= folds[i]["train_end"])]
    fold_validation = full_train_df[(full_train_df["TS"] >= folds[i]["valid_start"]) & (full_train_df["TS"] <= folds[i]["valid_end"])]
    folds[i]["nb_rows_train"] = len(fold_train)
    folds[i]["nb_rows_validation"] = len(fold_validation)
    folds[i]["pct_train_class_1"] = fold_train["class"].mean() * 100
    folds[i]["pct_validation_class_1"] = fold_validation["class"].mean() * 100

In [111]:
pd.DataFrame(folds)

,fold,train_start,train_end,valid_start,valid_end,nb_dates_train,nb_dates_validation,nb_rows_train,nb_rows_validation,pct_train_class_1,pct_validation_class_1
0,1,DATE_0001,DATE_2042,DATE_2043,DATE_2162,2042,120,428139,24114,50.480335,52.222775
1,2,DATE_0001,DATE_2162,DATE_2163,DATE_2282,2162,120,452253,24396,50.573241,50.483686
2,3,DATE_0001,DATE_2282,DATE_2283,DATE_2402,2282,120,476649,24764,50.568657,52.055403
3,4,DATE_0001,DATE_2402,DATE_2403,DATE_2522,2402,120,501413,25660,50.642085,52.209665


Les contrôles temporels des folds sont validés.

Chaque fold respecte l'ordre chronologique, aucune date n'est partagée entre train et validation, et chaque bloc de validation contient exactement 120 dates.

Les folds peuvent donc être utilisés comme protocole de validation temporelle pour les futurs modèles.